In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


In [18]:
def analyzeSpecificKernels(csvFile):
    df = pd.read_csv(csvFile)
    kernels = df.values.reshape(-1, 3, 3)


    sigmas = []
    for h in kernels:
        # To find the minimum magnitude on the unit circle
        mag = np.abs(np.fft.fft2(h, s=(32, 32)))
        sigmas.append(np.min(mag))

    sigmas = np.array(sigmas)
    idx_vulnerable = np.argmin(sigmas) # This is for Index 806
    idx_robust = np.argmax(sigmas)     # This is for Index 42

    targets = [
        {"idx": 0, "label": "Original_Sample"},
        {"idx": idx_vulnerable, "label": "Most_Vulnerable"},
        {"idx": idx_robust, "label": "Most_Robust"}
    ]
    
    for item in targets:
        idx = item["idx"]
        label = item["label"]
        h = kernels[idx]
        current_sigma = sigmas[idx]

        #Evaluate the Z-transform 
        gridSize = 100
        H = np.fft.fft2(h, s=(gridSize, gridSize))
        HShifted = np.fft.fftshift(H)
        magnitude = np.abs(HShifted)    

        omega = np.linspace(-np.pi, np.pi, gridSize)
        W1, W2 = np.meshgrid(omega, omega)
#plot the spatial kernel and its spectral response
        fig = plt.figure(figsize=(14, 6))
        
        
        ax1 = fig.add_subplot(121)
        im1 = ax1.imshow(h, cmap="viridis")
        ax1.set_title(f"Spatial Kernel {idx} ({label})")
        plt.colorbar(im1, ax=ax1)

        # Z-domain Surface
        ax2 = fig.add_subplot(122, projection="3d")
        surf = ax2.plot_surface(W1, W2, magnitude, cmap="magma", edgecolor="none")
        ax2.set_title(f"Spectral Response (Sigma: {current_sigma:.4f})")
        ax2.set_xlabel("$\omega_1$")
        ax2.set_ylabel("$\omega_2$")

       
        filename = f"kernel_{idx}_{label}.png"
        plt.savefig(filename)

        plt.close(fig)

<>:49: SyntaxWarning: invalid escape sequence '\o'
<>:50: SyntaxWarning: invalid escape sequence '\o'
<>:49: SyntaxWarning: invalid escape sequence '\o'
<>:50: SyntaxWarning: invalid escape sequence '\o'
/tmp/ipykernel_4704/3736634192.py:49: SyntaxWarning: invalid escape sequence '\o'
  ax2.set_xlabel("$\omega_1$")
/tmp/ipykernel_4704/3736634192.py:50: SyntaxWarning: invalid escape sequence '\o'
  ax2.set_ylabel("$\omega_2$")


In [19]:
if __name__ == "__main__":
    analyzeSpecificKernels("resnet18_layer1_3x3.csv")